# 0.05 Test site: Sentinel-2 download and preprocessing

**Purpose.** Test how achievable, how slow and how large it is to obtain Sentinel-2 MSI and NDVI input data for one
Geo-Wiki site (a 300 x 300 m area in Africa, i.e. the 100 x 100 m Geo-Wiki site plus a 100 m margin), so the cost for all
sites can be extrapolated. Second per-modality notebook after [0.04 Sentinel-1](0.04-tp-test-site-sentinel1.ipynb).

**Inputs.** Geo-Wiki site centre from `data/interim/geowiki/14871660/site_summary.csv` (notebook 0.03);
Google Earth Engine collection `COPERNICUS/S2_HARMONIZED` (Level-1C, not atmospherically corrected);
project-wide settings in `tomer_peled_mgi_thesis.config`.

**Outputs.** Raw GeoTIFF in `data/raw/sites/{site_id}/sentinel2/`, normalised Galileo-style array in
`data/processed/sites/{site_id}/`, timing/storage tables in `runs/tables/test_site/`, log in `runs/`.

**Method source.** The Earth Engine and preprocessing code is adapted from the Galileo repository
([nasaharvest/galileo](https://github.com/nasaharvest/galileo), MIT licence, commit `0f0b5b9`):
`src/data/earthengine/s2.py` (cloud score, shadow projection, quality score, quality mosaic, band selection),
`src/data/earthengine/eo.py` (`create_ee_image` time loop, `make_combine_bands_function`, `url`-mode download),
`src/data/earthengine/ee_bbox.py` (`EEBoundingBox`) and `src/data/dataset.py` (`_tif_to_array`, `_fillna`,
`calculate_ndi`, `Normalizer`, `save_h5py`). Deviations are marked `# changed:`. The temporal design follows the thesis
proposal: 30-day steps from July 2015 to July 2016.

```mermaid
flowchart LR
    A["site_summary.csv: site centre"] --> B["300 x 300 m bounding box (site + margin)"]
    B --> C["GEE COPERNICUS/S2_HARMONIZED: all images per 30-day step, closest scenes if empty"]
    C --> D["cloud score, shadow score, quality score per pixel"]
    D --> E["per-pixel best-quality mosaic, bands B2-B8A, B11, B12"]
    E --> F["getDownloadURL: GeoTIFF at 10 m, 12 steps x 10 bands"]
    F --> G["data/raw/sites/{site_id}/sentinel2/ .tif"]
    G --> H["fill NaN, reshape h w t c, add NDVI, normalise, float16"]
    H --> I["data/processed/sites/{site_id}/ .h5"]
    F -.timed.-> J["runs/tables/test_site/ timings and storage"]
    H -.timed.-> J
```

## 1. Settings

Project-wide values come from the package config (site id, dates, step length, site size, Earth Engine login
variables). Only notebook-specific choices are set here.

| Parameter | Default | Unit | Meaning |
| --- | --- | --- | --- |
| `TEST_SITE_ID` | 1151194 | Geo-Wiki sample id | Site under test (config) |
| `DATA_START_DATE`, `DATA_END_DATE` | 2015-07-01, 2016-07-01 | date | Study window (config) |
| `DAYS_PER_TIMESTEP` | 30 | days | Composite interval (config) |
| `SITE_SIZE_METRES` | 300 | m | Exported area edge length (config) |
| `N_DOWNLOAD_REPEATS` | 3 | runs | Repeated download requests, to see timing variability |
| `cloudThresh` | 0.2 | score 0-1 | Cloud-score threshold used to find shadows (Galileo) |
| `cloudHeights` | 200, 10000, 250 | m | Cloud heights for shadow projection (Galileo) |
| `irSumThresh`, `ndviThresh` | 0.3, -0.1 | reflectance sum, index | Dark-pixel and water thresholds (Galileo) |
| `erodePixels`, `dilationPixels` | 1.5, 3 | pixels | Morphological clean-up of shadow mask (Galileo) |
| `S2_DIV_VALUES` | 10000 | DN | Galileo's fixed S2 normalisation (reflectance 0-1); NDVI is unscaled |

In [ ]:
import time, shutil, math, warnings
from datetime import timedelta
from math import cos, radians

from googleapiclient.errors import HttpError
import ee, h5py, numpy as np, pandas as pd, rasterio, requests, xarray as xr
import rioxarray  # registers the rasterio engine
from einops import rearrange, repeat
from IPython.display import display

from tomer_peled_mgi_thesis import config as cfg
from tomer_peled_mgi_thesis.run_logging import log_run

SITE_DIR_RAW = cfg.RAW_DATA_DIR / "sites" / str(cfg.TEST_SITE_ID) / "sentinel2"
SITE_DIR_PROCESSED = cfg.PROCESSED_DATA_DIR / "sites" / str(cfg.TEST_SITE_ID)
TABLES_DIR = cfg.TABLES_DIR / "test_site"
N_DOWNLOAD_REPEATS = 3

# Galileo s2.py settings
S2_COLLECTION = "COPERNICUS/S2_HARMONIZED"
cloudThresh = 0.2
cloudHeights = [200, 10000, 250]
irSumThresh = 0.3
ndviThresh = -0.1
erodePixels = 1.5
dilationPixels = 3
ALL_S2_BANDS = ["B1", "B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B9", "B10", "B11", "B12"]
S2_BANDS = ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12"]  # B1, B9, B10 removed
S2_BANDS_OUT = S2_BANDS + ["NDVI"]  # NDVI is appended during preprocessing (Galileo dataset.py)
S2_SHIFT_VALUES = np.array([0.0] * len(S2_BANDS) + [0.0])
S2_DIV_VALUES = np.array([1e4] * len(S2_BANDS) + [1.0])

for d in (SITE_DIR_RAW, SITE_DIR_PROCESSED, TABLES_DIR):
    d.mkdir(parents=True, exist_ok=True)
stamp = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
log_ctx = log_run(cfg.RUNS_DIR / f"{stamp}-test-site-sentinel2.log")
logger = log_ctx.__enter__()  # closed in the summary cell
timings = []  # rows: step, phase, seconds

class timed:
    def __init__(self, step, phase): self.step, self.phase = step, phase
    def __enter__(self): self.t = time.perf_counter()
    def __exit__(self, *exc):
        secs = time.perf_counter() - self.t
        timings.append({"step": self.step, "phase": self.phase, "seconds": secs})
        logger.info("%s [%s]: %.2f s", self.step, self.phase, secs)

logger.info("Site=%s window=%s..%s step=%dd", cfg.TEST_SITE_ID, cfg.DATA_START_DATE, cfg.DATA_END_DATE, cfg.DAYS_PER_TIMESTEP)

## 2. Earth Engine login

Same as notebook 0.04: the `EE_PROJECT` / service-account variables from the config, otherwise the saved login from
`earthengine authenticate`. If the stored login is missing or invalid, `ee.Authenticate(force=True)` opens a one-time
browser login (not timed). No credential values are printed.

In [ ]:
# changed: credentials come from project config instead of Galileo's GCP_SA_KEY / hard-coded EE_PROJECT
credentials = "persistent"  # saved user login (None would mean no credentials)
if cfg.EE_SERVICE_ACCOUNT_KEY_FILE:
    credentials = ee.ServiceAccountCredentials(cfg.EE_SERVICE_ACCOUNT_EMAIL, cfg.EE_SERVICE_ACCOUNT_KEY_FILE)

def ee_initialise():
    ee.Initialize(credentials=credentials, project=cfg.EE_PROJECT,
                  opt_url="https://earthengine-highvolume.googleapis.com")
    ee.Number(1).getInfo()  # fails here, not later, if the login is invalid

try:
    ee_initialise()
except (ee.EEException, HttpError):  # a stale token surfaces as a 401 HttpError
    if credentials != "persistent":
        raise
    ee.Authenticate(force=True)  # one-time browser login; token is stored outside the project
with timed("Earth Engine initialise", "download"):
    ee_initialise()  # timed separately from any one-off login
print("Earth Engine initialised; login mode:", "persistent user credentials" if credentials == "persistent" else "service account")

## 3. Site geometry

As in notebook 0.04: the site centre is the Geo-Wiki site's mean location from notebook 0.03's cached summary, and a box
around it is built with Galileo's `EEBoundingBox.from_centre` metres-per-degree formulas. Galileo used 500 m
(1 km tiles); here it is half of `SITE_SIZE_METRES`.

In [ ]:
site = pd.read_csv(cfg.INTERIM_DATA_DIR / "geowiki" / "14871660" / "site_summary.csv",
                   usecols=["sampleid", "longitude", "latitude"]).set_index("sampleid").loc[cfg.TEST_SITE_ID]
lat, lon = float(site["latitude"]), float(site["longitude"])

def metre_per_degree(lat):  # Galileo ee_bbox.py
    m_lat = 111132.954 - 559.822 * cos(radians(2 * lat)) + 1.175 * cos(radians(4 * lat)) - 0.0023 * cos(radians(6 * lat))
    m_lon = 111412.84 * cos(radians(lat)) - 93.5 * cos(radians(3 * lat)) + 0.118 * cos(radians(5 * lat))
    return m_lat, m_lon

m_lat, m_lon = metre_per_degree(lat)
half = cfg.SITE_SIZE_METRES / 2  # changed: Galileo uses EXPORTED_HEIGHT_WIDTH_METRES / 2 = 500
min_lat, max_lat = lat - half / m_lat, lat + half / m_lat
min_lon, max_lon = lon - half / m_lon, lon + half / m_lon
polygon = ee.Geometry.Polygon([[[min_lon, min_lat], [min_lon, max_lat], [max_lon, max_lat], [max_lon, min_lat]]])
print(f"Site {cfg.TEST_SITE_ID}: lat={lat:.5f} lon={lon:.5f}")

## 4. Sentinel-2 availability per 30-day step

Galileo's Sentinel-2 code uses only scenes inside the step, so a step with no scene has nothing to composite and its
export would fail. Following the proposal's rule for Sentinel-1 ("if no acquisitions are made during the interval, the
closest available acquisition will be used instead"), this notebook applies the same fallback to Sentinel-2:
for an empty step, the scenes closest in time to the step midpoint (within 31 days either side, all scenes up to the
larger of the closest distance and 15 days) are used instead. This cell lists the Level-1C scenes over the site, counts
them per step and shows which scenes the fallback will pick (one small Earth Engine call, timed). The thesis proposal expects sparse early coverage because Sentinel-2A
only started in mid-2015 and Sentinel-2B in 2017.

In [ ]:
step_starts = []
cur, cur_end = cfg.DATA_START_DATE, cfg.DATA_START_DATE + timedelta(days=cfg.DAYS_PER_TIMESTEP)
while cur_end <= cfg.DATA_END_DATE:
    step_starts.append(cur)
    cur, cur_end = cur + timedelta(days=cfg.DAYS_PER_TIMESTEP), cur_end + timedelta(days=cfg.DAYS_PER_TIMESTEP)
n_timesteps = len(step_starts)

with timed("Query Sentinel-2 scene list", "download"):
    window = (ee.ImageCollection(S2_COLLECTION).filterBounds(polygon)
              .filterDate((cfg.DATA_START_DATE - timedelta(days=31)).isoformat(),
                          (cfg.DATA_END_DATE + timedelta(days=31)).isoformat()))
    scenes = pd.DataFrame({"date": pd.to_datetime(window.aggregate_array("system:time_start").getInfo(), unit="ms"),
                           "cloudy_pct": window.aggregate_array("CLOUDY_PIXEL_PERCENTAGE").getInfo()})
scenes["step"] = ((scenes.date.dt.date - cfg.DATA_START_DATE).apply(lambda d: d.days) // cfg.DAYS_PER_TIMESTEP)
in_window = scenes[scenes.step.between(0, n_timesteps - 1)]
per_step = (in_window.groupby("step").agg(scenes=("date", "size"), mean_cloudy_pct=("cloudy_pct", "mean"),
                                          min_cloudy_pct=("cloudy_pct", "min"))
            .reindex(range(n_timesteps)).fillna({"scenes": 0}))
per_step.insert(0, "step_start", step_starts)

def fallback_scenes(start):  # client-side preview of the closest-date rule used on Earth Engine below
    lo, hi = pd.Timestamp(start) - pd.Timedelta(days=31), pd.Timestamp(start) + pd.Timedelta(days=cfg.DAYS_PER_TIMESTEP + 31)
    near = scenes[(scenes.date >= lo) & (scenes.date < hi)]
    if near.empty:
        return pd.Series({"fallback_dates": "none within 31 days (will be NaN, then filled)", "fallback_days_from_mid": np.nan})
    dist = (near.date - (pd.Timestamp(start) + pd.Timedelta(days=cfg.DAYS_PER_TIMESTEP / 2))).abs()
    kept = near[dist <= max(dist.min(), pd.Timedelta(days=15))]
    return pd.Series({"fallback_dates": ", ".join(sorted({d.strftime("%Y-%m-%d") for d in kept.date})),
                      "fallback_days_from_mid": dist.min().total_seconds() / 86400})

empty = per_step.scenes == 0
per_step.loc[empty, ["fallback_dates", "fallback_days_from_mid"]] = per_step.loc[empty, "step_start"].apply(fallback_scenes).values
display(per_step.round(1))
print(len(in_window), "scenes in the window;", int(empty.sum()), "of", n_timesteps, "steps have no scene and use the closest-scene fallback")

## 5. Sentinel-2 composite graph (lazy, on Earth Engine)

Adapted from Galileo `s2.py` and `eo.create_ee_image`. For each 30-day step every scene intersecting the area is clipped
and given (1) a cloud score (brightness in blue and cirrus bands, visible brightness, moisture and snow tests), (2) a
shadow score (dark non-water pixels where a cloud's shadow should fall for three cloud heights, from the solar geometry)
and (3) a quality score, the negative smoothed maximum of cloud and shadow score. A per-pixel `qualityMosaic` then takes,
for each pixel, the values from the scene with the best quality score. Bands B1, B9 and B10 are used for the cloud score
and then dropped, leaving B2, B3, B4, B5, B6, B7, B8, B8A, B11, B12. The 12 composites are stacked as bands
`B2, B3, ..., B12, B2_1, ...`. This only builds the request; nothing is computed until download.

In [ ]:
def date_str(d): return d if isinstance(d, str) else d.strftime("%Y-%m-%d")  # Galileo utils.date_to_string

def rescale(img, exp, thresholds):  # Galileo s2.py
    return img.expression(exp, {"img": img}).subtract(thresholds[0]).divide(thresholds[1] - thresholds[0])

def computeS2CloudScore(img):  # Galileo s2.py
    toa = img.select(ALL_S2_BANDS).divide(10000)
    toa = toa.addBands(img.select(["QA60"]))
    score = ee.Image(1)
    score = score.min(rescale(toa, "img.B2", [0.1, 0.5]))  # clouds are bright in blue and cirrus bands
    score = score.min(rescale(toa, "img.B1", [0.1, 0.3]))
    score = score.min(rescale(toa, "img.B1 + img.B10", [0.15, 0.2]))
    score = score.min(rescale(toa, "img.B4 + img.B3 + img.B2", [0.2, 0.8]))  # bright in all visible bands
    ndmi = img.normalizedDifference(["B8", "B11"])  # clouds are moist
    score = score.min(rescale(ndmi, "img", [-0.1, 0.1]))
    ndsi = img.normalizedDifference(["B3", "B11"])  # clouds are not snow
    score = score.min(rescale(ndsi, "img", [0.8, 0.6]))
    score = score.max(ee.Image(0.001))
    score = score.reduceNeighborhood(reducer=ee.Reducer.mean(), kernel=ee.Kernel.square(5))
    return img.addBands(score.rename("cloudScore"))

def dilatedErossion(score):  # Galileo s2.py
    def erode(img, distance):
        d = img.Not().unmask(1).fastDistanceTransform(30).sqrt().multiply(ee.Image.pixelArea().sqrt())
        return img.updateMask(d.gt(distance))
    def dilate(img, distance):
        d = img.fastDistanceTransform(30).sqrt().multiply(ee.Image.pixelArea().sqrt())
        return d.lt(distance)
    score = score.reproject("EPSG:4326", None, 20)
    score = erode(score, erodePixels)
    score = dilate(score, dilationPixels)
    return score.reproject("EPSG:4326", None, 20)

def projectShadows(image):  # Galileo s2.py
    meanAzimuth = image.get("MEAN_SOLAR_AZIMUTH_ANGLE")
    meanZenith = image.get("MEAN_SOLAR_ZENITH_ANGLE")
    cloudMask = image.select(["cloudScore"]).gt(cloudThresh)
    darkPixelsImg = image.select(["B8", "B11", "B12"]).divide(10000).reduce(ee.Reducer.sum())
    ndvi = image.normalizedDifference(["B8", "B4"])
    waterMask = ndvi.lt(ndviThresh)
    darkPixels = darkPixelsImg.lt(irSumThresh)
    darkPixelMask = darkPixels.And(waterMask.Not()).And(cloudMask.Not())
    azR = ee.Number(meanAzimuth).add(180).multiply(math.pi).divide(180.0)
    zenR = ee.Number(meanZenith).multiply(math.pi).divide(180.0)

    def getShadows(cloudHeight):
        cloudHeight = ee.Number(cloudHeight)
        shadowCastedDistance = zenR.tan().multiply(cloudHeight)
        x = azR.sin().multiply(shadowCastedDistance).multiply(-1)
        y = azR.cos().multiply(shadowCastedDistance).multiply(-1)
        return image.select(["cloudScore"]).displace(ee.Image.constant(x).addBands(ee.Image.constant(y)))

    shadowMask = ee.ImageCollection.fromImages(ee.List(cloudHeights).map(getShadows)).mean()
    shadowMask = dilatedErossion(shadowMask.multiply(darkPixelMask))
    shadowScore = shadowMask.reduceNeighborhood(**{"reducer": ee.Reducer.max(), "kernel": ee.Kernel.square(1)})
    return image.addBands(shadowScore.rename(["shadowScore"]))

def computeQualityScore(img):  # Galileo s2.py
    score = img.select(["cloudScore"]).max(img.select(["shadowScore"]))
    score = score.reproject("EPSG:4326", None, 20).reduceNeighborhood(
        reducer=ee.Reducer.mean(), kernel=ee.Kernel.square(5), optimization="boxcar")
    return img.addBands(score.multiply(-1).rename("cloudShadowScore"))

def get_closest_dates(mid_date, imcol):  # Galileo utils.py (same function as in notebook 0.04)
    fifteen_days_in_ms = 1296000000
    mid_ms = ee.Date(date_str(mid_date)).millis()
    by_dist = imcol.map(lambda im: im.set("dateDist", ee.Number(im.get("system:time_start")).subtract(mid_ms).abs()))
    by_dist = by_dist.sort("dateDist", opt_ascending=True)
    max_diff = ee.Number(by_dist.first().get("dateDist")).max(ee.Number(fifteen_days_in_ms))
    return by_dist.filterMetadata("dateDist", "not_greater_than", max_diff)

def get_single_s2_image(region, start_date, end_date):  # Galileo s2.py
    s2 = ee.ImageCollection(S2_COLLECTION).filterBounds(region)
    in_step = s2.filterDate(date_str(start_date), date_str(end_date))
    # changed: Galileo uses only the scenes inside the step. Like Sentinel-1 (proposal), an empty step falls back to the
    # scenes closest to the step midpoint, searched within 31 days either side (the Sentinel-1 padding).
    padded = s2.filterDate(date_str(start_date - timedelta(days=31)), date_str(end_date + timedelta(days=31)))
    mid_date = start_date + (end_date - start_date) / 2
    closest = ee.ImageCollection(ee.Algorithms.If(padded.size().gt(0), get_closest_dates(mid_date, padded), padded))
    raw = ee.ImageCollection(ee.Algorithms.If(in_step.size().gt(0), in_step, closest))
    imgC = (raw.map(lambda x: x.clip(region)).map(lambda x: x.set("ROI", region))
            .map(computeS2CloudScore).map(projectShadows).map(computeQualityScore).sort("CLOUDY_PIXEL_PERCENTAGE"))
    cloudFree = imgC.qualityMosaic("cloudShadowScore").select(S2_BANDS).toDouble()  # double, like Sentinel-1
    # changed: Galileo fails when no scene is found at all (select on a band-less mosaic). Return an all-NaN step instead;
    # preprocessing then fills it like any other missing value (Galileo _fillna).
    all_nan = ee.Image.cat([ee.Image(0).divide(0)] * len(S2_BANDS)).rename(S2_BANDS).toDouble()
    return ee.Image(ee.Algorithms.If(raw.size().gt(0), cloudFree, all_nan))

def make_combine_bands_function(bands):  # Galileo eo.py (author: Jamie Vleeshouwer)
    def combine_bands(current, previous):
        previous = ee.Image(previous)
        current = current.select(bands)
        return ee.Algorithms.If(ee.Algorithms.IsEqual(previous, None), current, previous.addBands(ee.Image(current)))
    return combine_bands

def create_s2_ee_image(region, start_date, end_date, days_per_timestep):  # Galileo eo.create_ee_image, S2 only
    images, cur, cur_end = [], start_date, start_date + timedelta(days=days_per_timestep)
    while cur_end <= end_date:
        images.append(get_single_s2_image(region, cur, cur_end))
        cur, cur_end = cur + timedelta(days=days_per_timestep), cur_end + timedelta(days=days_per_timestep)
    return ee.Image(ee.ImageCollection(images).iterate(make_combine_bands_function(S2_BANDS)))

with timed("Build Earth Engine graph", "download"):
    ee_image = create_s2_ee_image(polygon, cfg.DATA_START_DATE, cfg.DATA_END_DATE, cfg.DAYS_PER_TIMESTEP)
print("Timesteps:", n_timesteps, "-> expected bands:", n_timesteps * len(S2_BANDS))

## 6. Download

As in notebook 0.04 and Galileo's `url` mode: `getDownloadURL` requests a GeoTIFF at 10 m scale (Earth Engine resamples the
20 m bands B5-B7, B8A, B11, B12 to 10 m, nearest neighbour by default) and the response is streamed to disk. The request is
repeated `N_DOWNLOAD_REPEATS` times to expose latency variability; the cloud scoring runs on Earth Engine's servers, so it
is part of the request time. Bands are labelled `<band>_<step start date>` after the download; that is the only edit to the
raw file.

In [ ]:
tif_path = SITE_DIR_RAW / f"{cfg.TEST_SITE_ID}_sentinel2_{cfg.DATA_START_DATE}_{cfg.DATA_END_DATE}.tif"
download_secs = []
for i in range(N_DOWNLOAD_REPEATS):
    t0 = time.perf_counter()
    url = ee_image.getDownloadURL({"region": polygon, "scale": 10, "filePerBand": False, "format": "GEO_TIFF"})
    t_url = time.perf_counter()
    r = requests.get(url, stream=True)
    r.raise_for_status()
    with tif_path.open("wb") as f:
        shutil.copyfileobj(r.raw, f)
    t1 = time.perf_counter()
    timings += [{"step": f"Request download URL (run {i+1})", "phase": "download", "seconds": t_url - t0},
                {"step": f"Transfer GeoTIFF (run {i+1})", "phase": "download", "seconds": t1 - t_url}]
    download_secs.append(t1 - t0)
    logger.info("Download run %d: url %.2f s, transfer %.2f s, %d bytes", i + 1, t_url - t0, t1 - t_url, tif_path.stat().st_size)
print("Download seconds per run:", [round(s, 2) for s in download_secs])

# changed: Galileo's GeoTIFFs have no band names; label each band "<band>_<step start date>" (t-major, c-minor order)
band_names = [f"{b}_{d}" for d in step_starts for b in S2_BANDS]
with rasterio.open(tif_path, "r+") as src:
    src.descriptions = band_names
    print(src.count, "bands:", src.descriptions[:3], "...", src.descriptions[-2:])

## 7. Preprocessing

Mirrors Galileo's `Dataset._tif_to_array`: read the GeoTIFF, reshape `(t c) h w -> h w t c`, replace NaN/Inf with the
per-band mean (`_fillna`), add NDVI = (B8 - B4) / (B8 + B4) from the raw band values (0 where the sum is not positive,
Galileo's `calculate_ndi`), normalise (bands divided by 10,000, NDVI unchanged), cast to float16 and save to HDF5
(`save_h5py`). `changed:` the Sentinel-1/climate parts are omitted (other notebooks), and the fixed shift/div values are
used (Galileo's `std=False` path) since the `normalization.json` mean/std file is not available for this dataset.
The month index array comes from the study start date as in `month_array_from_file`.

Steps with no scene even after the closest-scene fallback are written by Earth Engine as zeros; they are detected (all values exactly 0) and set to NaN first,
so `_fillna` replaces them. Note that Galileo's `_fillna` fills with the mean of the per-step band means, counting a
missing step as 0 in that mean, so the filled values are slightly low (by about 1/12 here) and they mix seasons; the
step carries no real information and the filled-step count should be reported with any results.

In [ ]:
def fillna(data, n_bands):  # Galileo dataset.py _fillna (4D branch)
    if data.shape[-1] != n_bands:
        raise ValueError(f"Expected {n_bands} bands - got {data.shape[-1]}")
    is_bad = np.isnan(data) | np.isinf(data)
    if not is_bad.any():
        return data
    data = np.nan_to_num(data, nan=np.nan, posinf=np.nan, neginf=np.nan)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        mean_per_time_band = np.nanmean(data, axis=(0, 1))  # t, b
    mean_per_time_band = np.nan_to_num(mean_per_time_band, nan=0, posinf=0, neginf=0)
    fill = repeat(np.nanmean(mean_per_time_band, axis=0), "b -> h w t b",
                  h=data.shape[0], w=data.shape[1], t=data.shape[2]) * is_bad
    return np.nan_to_num(data, nan=0, posinf=0, neginf=0) + fill

def calculate_ndi(x, band_1, band_2):  # Galileo dataset.py calculate_ndi; x is [h, w, t, bands] in S2_BANDS order
    b1, b2 = x[..., S2_BANDS.index(band_1)], x[..., S2_BANDS.index(band_2)]
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="invalid value encountered in divide")
        return np.expand_dims(np.where((b1 + b2) > 0, (b1 - b2) / (b1 + b2), 0), -1)

with timed("Read GeoTIFF", "preprocess"):
    with xr.open_dataarray(tif_path, engine="rasterio") as da:  # Galileo uses rioxarray.open_rasterio
        values = da.values  # [bands, H, W]
with timed("Reshape, fill NaN, NDVI, normalise, float16", "preprocess"):
    s2 = rearrange(values, "(t c) h w -> h w t c", c=len(S2_BANDS))
    # changed: Earth Engine writes the all-NaN step of a scene-less search window as exact zeros in the GeoTIFF, which real
    # reflectance never is for a whole step; mark those steps as NaN so Galileo's fill below handles them
    empty_steps = np.flatnonzero((s2 == 0).all(axis=(0, 1, 3)))
    s2[:, :, empty_steps, :] = np.nan
    n_nan = int(np.isnan(s2).sum())
    s2 = fillna(s2, len(S2_BANDS))
    s2 = np.concatenate((s2, calculate_ndi(s2, "B8", "B4")), axis=-1)
    s2 = ((s2 - S2_SHIFT_VALUES) / S2_DIV_VALUES).astype(np.half)
    months = np.fmod(np.arange(cfg.DATA_START_DATE.month - 1, cfg.DATA_START_DATE.month - 1 + s2.shape[2]), 12)
h5_path = SITE_DIR_PROCESSED / "sentinel2.h5"
with timed("Save HDF5", "preprocess"):
    with h5py.File(h5_path, "w") as hf:  # Galileo save_h5py (S2 + NDVI only; dataset name kept as s_t_x)
        hf.create_dataset("s_t_x", data=s2)
        hf.create_dataset("months", data=months)
print(f"Array [h, w, t, c] = {s2.shape}, dtype {s2.dtype}; empty steps filled: {empty_steps.tolist()}; NaN cells before fill: {n_nan}; months: {months.tolist()}")

## 8. Timing and storage summary

Totals split into download (login, scene query, graph, URL request, transfer) and preprocessing. Storage is the on-disk
size of the raw GeoTIFF and the processed HDF5. The extrapolation multiplies this single-site, sequential result by a
number of sites; it ignores parallel requests, Earth Engine quota limits and site-to-site variation, so treat it as a
rough scale only.

In [ ]:
t = pd.DataFrame(timings)
first_download = t[t.step.str.contains("run 1|initialise|graph|scene list")].seconds.sum()
mean_repeat = float(np.mean(download_secs))
pre = t[t.phase == "preprocess"].seconds.sum()
storage = pd.DataFrame({"file": [tif_path.name, h5_path.name], "kind": ["raw GeoTIFF", "processed HDF5"],
                        "bytes": [tif_path.stat().st_size, h5_path.stat().st_size]})
storage["MB"] = storage.bytes / 1e6
summary = pd.DataFrame([
    {"metric": "Download incl. login, scene query, graph, first request (s)", "value": first_download},
    {"metric": "Mean download per request (s)", "value": mean_repeat},
    {"metric": "Preprocessing (s)", "value": pre},
    {"metric": "Per-site time, steady state (s)", "value": mean_repeat + pre},
    {"metric": "Raw GeoTIFF (MB)", "value": storage.MB[0]},
    {"metric": "Processed HDF5 (MB)", "value": storage.MB[1]},
])
display(t.round(3)); display(storage[["kind", "bytes", "MB"]].round(4)); display(summary.round(3))
N = 1000
per_site = mean_repeat + pre
print(f"Extrapolation per {N} sites, sequential: {per_site * N / 3600:.2f} h; "
      f"raw {storage.MB[0] * N / 1e3:.2f} GB; processed {storage.MB[1] * N / 1e3:.2f} GB")
t.assign(site_id=cfg.TEST_SITE_ID, modality="sentinel2").to_csv(TABLES_DIR / f"{cfg.TEST_SITE_ID}_sentinel2_timings.csv", index=False)
storage.assign(site_id=cfg.TEST_SITE_ID, modality="sentinel2").to_csv(TABLES_DIR / f"{cfg.TEST_SITE_ID}_sentinel2_storage.csv", index=False)
per_step.assign(site_id=cfg.TEST_SITE_ID).to_csv(TABLES_DIR / f"{cfg.TEST_SITE_ID}_sentinel2_scenes_per_step.csv", index=False)
logger.info("Per-site steady-state %.2f s; raw %d B; processed %d B", per_site, storage.bytes[0], storage.bytes[1])
log_ctx.__exit__(None, None, None);

## 9. Conclusions

Result of the run on 2026-10-29 (single site, one machine, sequential; timings vary between requests):

- **Achievable, with a gap handled by fallback.** Earth Engine returned the composites (120 bands: 12 steps x 10 bands,
  31 x 31 pixels at 10 m) and the NDVI channel was added in preprocessing. One of 12 steps (31 July to 30 August 2015) has
  no Sentinel-2 scene over the site (52 scenes in the 12 steps), which Galileo's code cannot handle. Following the
  Sentinel-1 rule in the proposal, it uses the scenes closest in time instead: the 12 September 2015 scenes, 28 days from
  the step midpoint. That step is therefore largely a repeat of the next step (31 July to 30 August uses September
  scenes, and 30 August to 29 September contains them too), so its temporal information is weaker than the other steps.
  Expect empty steps at other sites, especially in 2015 when only Sentinel-2A was operating; if no scene exists within 31
  days the step is exported as NaN and filled during preprocessing (not triggered here).
- **Values.** Surface-like reflectance (B2 about 0.10-0.19, B8 about 0.20-0.33) and NDVI 0.16-0.57 per step, which is
  plausible for a savanna/cropland site with a dry-season dip. Cloud masking quality was not inspected visually.
- **Time.** About 6-8 s per site for download (request about 1.5-2.5 s, transfer about 3.5-6 s; the cloud scoring runs on
  Earth Engine) plus about 0.5 s preprocessing, plus a one-off 2 s login and 2 s scene query, so roughly 1.7-2.2 h per
  1,000 sites if run one after another, about 1.5-2 times Sentinel-1.
- **Storage.** About 281 KB raw GeoTIFF and 256 KB processed float16 HDF5 per site, i.e. about 0.28 GB and 0.26 GB per
  1,000 sites. The processed file is nearly as large as the raw one because it holds 11 channels (including NDVI) as float16.
- **Not yet assessed.** Request limits for many sites, parallel downloads, and variation between sites and regions.